# Moon phase vs crime/accident (sklearn)
Fetch Supabase data, merge moon phases with daily outcomes, and quantify relationships via linear models.

In [ ]:
import os
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()
if not os.getenv("SUPABASE_URL") or not os.getenv("SUPABASE_KEY"):
    raise SystemExit("Missing SUPABASE_URL or SUPABASE_KEY in environment")

supabase = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_KEY"])

def fetch_table(name: str, batch_size: int = 1000, limit: int | None = None) -> pd.DataFrame:
    rows, start = [], 0
    while True:
        end = start + batch_size - 1
        resp = supabase.table(name).select("*").range(start, end).execute()
        batch = resp.data or []
        rows.extend(batch)
        if len(batch) < batch_size or (limit and len(rows) >= limit):
            break
        start += batch_size
    if limit:
        rows = rows[:limit]
    return pd.DataFrame(rows)


In [2]:
crimes_df = fetch_table("chicago_crimes")
accident_df = fetch_table("chicago_accident_cleaned")
moon_df = fetch_table("cleaned_moon_data")

print(f"Crimes: {len(crimes_df)} rows")
print(f"Accidents: {len(accident_df)} rows")
print(f"Moon: {len(moon_df)} rows")

if crimes_df.empty or accident_df.empty or moon_df.empty:
    raise SystemExit("One or more tables are empty; cannot proceed")


Crimes: 251337 rows
Accidents: 110720 rows
Moon: 1826 rows


In [9]:

crimes_daily = crimes_df.groupby("date_only")["crime_count"].sum().reset_index().rename(columns={"date_only": "date"})
acc_daily = accident_df.groupby("crash_date")["incidents_count"].sum().reset_index().rename(columns={"crash_date": "date"})

crimes_daily["date"] = pd.to_datetime(crimes_daily["date"])
acc_daily["date"] = pd.to_datetime(acc_daily["date"])
moon_df = moon_df.copy()
moon_df["date"] = pd.to_datetime(moon_df["date"])

merged = moon_df.merge(crimes_daily, on="date", how="left").merge(acc_daily, on="date", how="left")
merged = merged.dropna(subset=["avg_phase", "moon_category"]).copy()


In [19]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["crime_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= crime_count =========")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nTop positive weights:\n", coefs.tail(5))
print("\nTop negative weights:\n", coefs.head(5))



========= crime_count =========

5-fold R^2: mean=-0.860, std=0.713

Top positive weights:
 moon_First Quarter    -4.593703
moon_New Moon         -2.795000
avg_phase             -0.112031
moon_Waxing Gibbous    4.214824
moon_Full Moon         8.426388
dtype: float64

Top negative weights:
 moon_Waxing Crescent   -5.252509
moon_First Quarter     -4.593703
moon_New Moon          -2.795000
avg_phase              -0.112031
moon_Waxing Gibbous     4.214824
dtype: float64


In [20]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["incidents_count"].fillna(0)

model = LinearRegression()
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= incidents_count =========")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nTop positive weights:\n", coefs.tail(5))
print("\nTop negative weights:\n", coefs.head(5))


========= incidents_count =========

5-fold R^2: mean=-0.032, std=0.019

Top positive weights:
 moon_Waxing Gibbous    -1.450851
avg_phase               0.095590
moon_First Quarter      1.462209
moon_New Moon           2.203138
moon_Waxing Crescent    2.922134
dtype: float64

Top negative weights:
 moon_Full Moon        -5.136631
moon_Waxing Gibbous   -1.450851
avg_phase              0.095590
moon_First Quarter     1.462209
moon_New Moon          2.203138
dtype: float64


In [23]:
from sklearn.ensemble import RandomForestRegressor

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["crime_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on crime_count ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nTop drivers (importance):\n", importances.tail(5))


=== Random forest on crime_count ===

5-fold R^2: mean=-1.171, std=0.873

Top drivers (importance):
 moon_Waxing Gibbous     0.000531
moon_First Quarter      0.001518
moon_Waxing Crescent    0.002739
moon_New Moon           0.004645
avg_phase               0.990372
dtype: float64


In [24]:
from sklearn.ensemble import RandomForestRegressor

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["incidents_count"].fillna(0)

model = RandomForestRegressor(
    n_estimators=300, random_state=42, min_samples_leaf=5, n_jobs=-1
)
r2 = cross_val_score(model, X, y, cv=5, scoring="r2")
model.fit(X, y)
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()

print(f"\n=== Random forest on incidents_count ===")
print(f"\n5-fold R^2: mean={r2.mean():.3f}, std={r2.std():.3f}")
print("\nTop drivers (importance):\n", importances.tail(5))


=== Random forest on incidents_count ===

5-fold R^2: mean=-0.219, std=0.065

Top drivers (importance):
 moon_Waxing Crescent    0.000534
moon_Waxing Gibbous     0.000649
moon_Full Moon          0.000755
moon_First Quarter      0.001208
avg_phase               0.996704
dtype: float64


In [27]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import make_scorer, mean_poisson_deviance

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["crime_count"].fillna(0)
model = PoissonRegressor(max_iter=300)
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False)
dev = -cross_val_score(model, X, y, cv=5, scoring=scorer)
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========= Poisson regression on crime_count =========")
print(f"\n5-fold mean Poisson deviance: mean={dev.mean():.3f}, std={dev.std():.3f}")
print("\nLargest positive effects:\n", coefs.tail(5))
print("\nLargest negative effects:\n", coefs.head(5))


========= Poisson regression on crime_count =========

5-fold mean Poisson deviance: mean=37.690, std=39.277

Largest positive effects:
 moon_First Quarter    -0.006944
moon_New Moon         -0.003239
avg_phase             -0.000150
moon_Waxing Gibbous    0.005859
moon_Full Moon         0.011779
dtype: float64

Largest negative effects:
 moon_Waxing Crescent   -0.007454
moon_First Quarter     -0.006944
moon_New Moon          -0.003239
avg_phase              -0.000150
moon_Waxing Gibbous     0.005859
dtype: float64


In [28]:
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import make_scorer, mean_poisson_deviance

X = pd.concat(
    [
        merged[["avg_phase"]],
        pd.get_dummies(merged["moon_category"], prefix="moon", dummy_na=False),
    ],
    axis=1,
)
y = merged["incidents_count"].fillna(0)
model = PoissonRegressor(max_iter=300)
scorer = make_scorer(mean_poisson_deviance, greater_is_better=False)
dev = -cross_val_score(model, X, y, cv=5, scoring=scorer)
model.fit(X, y)
coefs = pd.Series(model.coef_, index=X.columns).sort_values()

print(f"\n========== Poisson regression on incidents_count ==========")
print(f"\n5-fold mean Poisson deviance: mean={dev.mean():.3f}, std={dev.std():.3f}")
print("\nLargest positive effects:\n", coefs.tail(5))
print("\nLargest negative effects:\n", coefs.head(5))



========== Poisson regression on incidents_count ==========

5-fold mean Poisson deviance: mean=9.315, std=6.228

Largest positive effects:
 moon_Waxing Gibbous    -0.005296
avg_phase               0.000638
moon_New Moon           0.005447
moon_First Quarter      0.013540
moon_Waxing Crescent    0.020269
dtype: float64

Largest negative effects:
 moon_Full Moon        -0.033970
moon_Waxing Gibbous   -0.005296
avg_phase              0.000638
moon_New Moon          0.005447
moon_First Quarter     0.013540
dtype: float64
